# 02 — Identificação dos dados MODIS × VIIRS (skill *identificador-de-dados*)
Cada conclusão aqui segue a regra: **o que foi identificado, por qual critério, com qual evidência e até onde vale.**

Escopo: arquivos anuais por país do NASA FIRMS, **Brasil**. MODIS C6.1 (Terra + Aqua) 2001–2025 e VIIRS S-NPP 375 m 2012–2025.
Fontes: `data/raw/*.parquet` (colunas reduzidas) + trechos dos CSVs originais (todas as colunas).

In [1]:
import io, requests, duckdb, numpy as np, pandas as pd
from pathlib import Path
RAW = Path('../data/raw'); con = duckdb.connect(); q = lambda s: con.execute(s).df()
M = f"read_parquet('{RAW.as_posix()}/modis_*.parquet')"
V = f"read_parquet('{RAW.as_posix()}/viirs-snpp_*.parquet')"
BASE = 'https://firms.modaps.eosdis.nasa.gov/data/country'
def csv_original(fonte, ano, nbytes=2_000_000):
    '''Trecho do CSV original, tudo como texto (preserva zeros à esquerda e casas decimais).'''
    t = requests.get(f'{BASE}/{fonte}/{ano}/{fonte}_{ano}_Brazil.csv', headers={'Range': f'bytes=0-{nbytes}'}, timeout=60).text
    return pd.read_csv(io.StringIO(t[:t.rfind('\n')]), dtype=str)
om, ov = csv_original('modis', 2020), csv_original('viirs-snpp', 2020)

---
## Passo 1 — O que cada registro representa
**Critério:** um registro = uma linha do CSV. Hipótese a testar: *uma linha = um pixel quente numa passagem do satélite* (não um incêndio).

### 1a. Tamanho real do pixel (`scan` × `track`, km)

In [2]:
px = pd.DataFrame({s: (d.scan.astype(float) * d.track.astype(float)).describe(percentiles=[.05,.5,.95]).round(3)
                   for s, d in [('MODIS km²', om), ('VIIRS km²', ov)]})
px

,MODIS km²,VIIRS km²
count,25047.000,25078.000
mean,2.159,0.220
std,1.643,0.084
min,1.000,0.137
5%,1.000,0.140
50%,1.430,0.198
95%,5.780,0.386
max,9.600,0.624


**Leitura:** a área do pixel **varia dentro do próprio sensor** (cresce na borda da faixa de varredura). Então nem o MODIS é sempre 1 km nem o VIIRS sempre 375 m — esses são valores **no nadir**.
**Consequência:** um ponto representa uma área diferente conforme o sensor *e* a posição na faixa → contar pontos ≠ medir área.

### 1b. Uma linha é um incêndio? Teste num dia e região (2020-09-13, Aqua × S-NPP, passagem da tarde)

In [3]:
box = 'latitude BETWEEN -12 AND -10 AND longitude BETWEEN -52 AND -50'
a = q(f"SELECT latitude, longitude, acq_time FROM {M} WHERE acq_date='2020-09-13' AND satellite='Aqua' AND daynight='D' AND type=0 AND {box}")
v = q(f"SELECT latitude, longitude, acq_time FROM {V} WHERE acq_date='2020-09-13' AND daynight='D' AND type=0 AND {box}")
def dist_km(p, r):   # distância aproximada (km) de cada ponto de p ao ponto mais próximo de r
    dy = (p.latitude.values[:, None] - r.latitude.values[None]) * 111.0
    dx = (p.longitude.values[:, None] - r.longitude.values[None]) * 111.0 * np.cos(np.radians(p.latitude.values[:, None]))
    return np.sqrt(dx**2 + dy**2).min(1)
d_av, d_va = dist_km(a, v), dist_km(v, a)
print(f'MODIS Aqua: {len(a)} pontos | VIIRS: {len(v)} pontos  (razão {len(v)/len(a):.2f})')
for tol in [0.5, 1, 2, 5]:
    print(f'tolerância {tol} km → MODIS com VIIRS perto: {np.mean(d_av<=tol):5.1%} | VIIRS com MODIS perto: {np.mean(d_va<=tol):5.1%}')

MODIS Aqua: 236 pontos | VIIRS: 981 pontos  (razão 4.16)
tolerância 0.5 km → MODIS com VIIRS perto: 55.9% | VIIRS com MODIS perto: 29.5%
tolerância 1 km → MODIS com VIIRS perto: 92.8% | VIIRS com MODIS perto: 69.7%
tolerância 2 km → MODIS com VIIRS perto: 99.2% | VIIRS com MODIS perto: 85.9%
tolerância 5 km → MODIS com VIIRS perto: 100.0% | VIIRS com MODIS perto: 94.7%


**Critério de correspondência:** dois pontos são candidatos ao *mesmo evento* se a distância ≤ tolerância. A tolerância **não é arbitrária**: testamos 0,5 / 1 / 2 / 5 km, cobrindo do pixel VIIRS ao pixel MODIS na borda da faixa (~4,8 km).
**Leitura esperada:** quase todo ponto MODIS tem VIIRS perto (mesmo fogo), mas muitos VIIRS não têm MODIS perto (fogos pequenos que o MODIS não detecta, ou vários pixels VIIRS dentro de um pixel MODIS).
**Conclusão:** 1 linha = 1 **pixel** detectado, não 1 incêndio. Por isso a contagem bruta não é comparável entre sensores.

### 1c. Diferença de horário entre pontos correspondentes (mesmo evento)

In [4]:
t = lambda s: s.astype(int) // 100 * 60 + s.astype(int) % 100
ia = dist_km(v, a) <= 1
j = np.argmin(((v.latitude.values[:, None]-a.latitude.values[None])**2 + (v.longitude.values[:, None]-a.longitude.values[None])**2), 1)
dif = (t(v.acq_time).values - t(a.acq_time).values[j])[ia]
print(f'pares a ≤1 km: {ia.sum()} | diferença de horário VIIRS − Aqua (min): mediana {np.median(dif):.0f}, p5 {np.percentile(dif,5):.0f}, p95 {np.percentile(dif,95):.0f}')

pares a ≤1 km: 684 | diferença de horário VIIRS − Aqua (min): mediana 4, p5 4, p95 4


**Leitura:** Aqua e S-NPP veem o mesmo fogo com poucos minutos de diferença → é o par adequado para calibrar a conversão.

---
## Passo 2 — Critério de comparação
| Nível | Pergunta | Onde testamos |
|---|---|---|
| **Valores** | O campo tem o mesmo formato/unidade/regra? | Passos 3 e 5 |
| **Registros** | Uma linha MODIS e uma VIIRS podem ser o mesmo registro? | **Não**: vêm de instrumentos e produtos diferentes; não existe chave comum |
| **Evento** | Representam o mesmo fogo? | Passo 1b (proximidade espaço-tempo) |

---
## Passo 3 — Campos: domínio de valores observado
### 3a. Tipos e domínios

In [5]:
def perfil(d, nome):
    return pd.DataFrame({c: [nome, d[c].nunique(), ', '.join(sorted(d[c].unique())[:4]) + (' …' if d[c].nunique() > 4 else '')] for c in d.columns},
                        index=['sensor', 'nº valores distintos', 'exemplos']).T
pd.concat([perfil(om, 'MODIS'), perfil(ov, 'VIIRS')], axis=1)

,sensor,nº valores distintos,exemplos,sensor,nº valores distintos,exemplos
latitude,MODIS,23295,"-0.0041, -0.0191, -0.0638, -0.0791 …",VIIRS,24836,"-0.04906, -0.05023, -0.05255, -0.0537 …"
longitude,MODIS,22866,"-34.8234, -34.8514, -34.8577, -34.8579 …",VIIRS,24663,"-34.8022, -34.82267, -34.82626, -34.82671 …"
brightness,MODIS,797,"300, 300.1, 300.2, 300.3 …",NaN,NaN,NaN
scan,MODIS,39,"1, 1.1, 1.2, 1.3 …",VIIRS,49,"0.32, 0.33, 0.34, 0.35 …"
track,MODIS,11,"1, 1.1, 1.2, 1.3 …",VIIRS,43,"0.36, 0.37, 0.38, 0.39 …"
acq_date,MODIS,109,"2020-01-01, 2020-01-02, 2020-01-03, 2020-01-04 …",VIIRS,37,"2020-01-01, 2020-01-02, 2020-01-03, 2020-01-04 …"
acq_time,MODIS,610,"0015, 0020, 0021, 0026 …",VIIRS,394,"0245, 0251, 0254, 0255 …"
satellite,MODIS,2,"Aqua, Terra",VIIRS,1,N
instrument,MODIS,1,MODIS,VIIRS,1,VIIRS
confidence,MODIS,99,"0, 10, 100, 11 …",VIIRS,3,"h, l, n"


### 3b. Versão do produto ao longo dos anos (mudança de método?)

In [6]:
ver = []
for f, anos in [('modis', [2001, 2006, 2012, 2018, 2025]), ('viirs-snpp', [2012, 2018, 2025])]:
    for ano in anos:
        d = csv_original(f, ano, 300_000); ver.append((f, ano, ', '.join(sorted(d.version.unique()))))
pd.DataFrame(ver, columns=['fonte', 'ano', 'version'])

,fonte,ano,version
0,modis,2001,6.2
1,modis,2006,6.2
2,modis,2012,6.2
3,modis,2018,6.03
4,modis,2025,61.03
5,viirs-snpp,2012,2
6,viirs-snpp,2018,2
7,viirs-snpp,2025,2


**Resultado:** VIIRS mostra `2` em todos os anos testados → série internamente consistente.
MODIS mostra códigos **diferentes** entre anos (`6.2`, `6.03`, `61.03`). **Classificação: inconclusivo.** Pode ser só o formato do código de versão ou mistura de coleções/processamentos no arquivo.
**O que falta para concluir:** ler a coluna `version` do arquivo inteiro de cada ano (não só o começo) e conferir na documentação do MCD14ML o significado de cada código. Se houver mistura de coleções, é um possível ponto de quebra na série MODIS.

### 3c. `confidence`: domínio e relação com dia/noite

In [7]:
cm_ = q(f"SELECT daynight, min(confidence::INT) min, max(confidence::INT) max, round(avg(confidence::INT),1) media FROM {M} GROUP BY 1")
cv_ = q(f"SELECT daynight, confidence, count(*) n FROM {V} GROUP BY ALL ORDER BY 1,2")
cv_['%'] = (100*cv_.n/cv_.groupby('daynight').n.transform('sum')).round(1)
display(cm_); cv_.pivot(index='confidence', columns='daynight', values='%')

,daynight,min,max,media
0,N,0,100,69.8
1,D,0,100,68.2


daynight,D,N
confidence,,
h,9.1,0.9
l,12.8,NaN
n,78.1,99.1


**Leitura:** MODIS é um **número contínuo** (0–100); VIIRS é **categoria** (l/n/h) e a regra muda entre dia e noite (segundo o FAQ do FIRMS, *high* está ligado a pixels saturados).
**Classificação:** *não comparáveis diretamente* como valor. Só dá para usar como **filtro** com um critério explícito (MODIS ≥ 30 ≈ VIIRS n|h), cuja equivalência é **hipótese** — verificável comparando as contagens filtradas no período de sobreposição.

### 3d. `frp` (MW): zeros, negativos, extremos

In [8]:
q(f'''SELECT 'MODIS' s, count(*) n, sum((frp IS NULL)::INT) nulos, sum((frp=0)::INT) zeros, sum((frp<0)::INT) negativos, min(frp) min, max(frp) max FROM {M}
      UNION ALL SELECT 'VIIRS', count(*), sum((frp IS NULL)::INT), sum((frp=0)::INT), sum((frp<0)::INT), min(frp), max(frp) FROM {V}''')

,s,n,nulos,zeros,negativos,min,max
0,MODIS,9966890,0.0,539.0,2.0,-21.00,13290.90
1,VIIRS,19229379,0.0,3692.0,61.0,-3.55,8014.33


**Leitura:** FRP tem a mesma unidade (MW), mas é a potência **do pixel**. Como o pixel VIIRS é ~7× menor, o FRP por ponto é menor e o VIIRS registra fogos fracos que o MODIS não detecta.
**Classificação:** *equivalente sob critério* só se somado por área/evento; **por ponto, não comparável**.
**Valores inválidos:** FRP negativo não tem significado físico (MODIS 2 registros, VIIRS 61). São poucos; não entram em somas de FRP, mas a detecção é mantida na contagem.

---
## Passo 4 — Por que diferem (observado × causa)
| Diferença observada | Evidência | Causa | Status |
|---|---|---|---|
| VIIRS tem ~3,8× mais pontos | `grid_monthly`: razão anual 3,6–4,0 (2012–2022) | Pixel menor + maior sensibilidade | **Comprovada em parte** (1b mostra vários VIIRS por MODIS); a fração de fogo pequeno é hipótese |
| Razão cai para ~3,4 em 2025 | razão anual | Deriva da órbita do Terra/Aqua | **Comprovada a deriva** (teste abaixo); que ela *cause* a queda da razão continua hipótese |
| `type=3` muito maior no VIIRS | 133 × 29.251 | Pixel menor detecta flares pequenos no mar | **Hipótese** — testar: mapear os pontos type=3 |
| Terra só até jul/2002 sem Aqua | início das séries | Lançamento do Aqua (jul/2002) | **Comprovada** pelas datas |

In [9]:
# Teste da hipótese da deriva orbital: hora solar local média por ano (só passagens diurnas)
q(f'''SELECT year(CAST(acq_date AS DATE)) ano, satellite,
        round(avg(((acq_time::INT//100) + (acq_time::INT%100)/60 + longitude/15 + 24) % 24), 2) hora_local_media
      FROM {M} WHERE daynight='D' GROUP BY ALL ORDER BY 1''').pivot(index='ano', columns='satellite', values='hora_local_media')

satellite,Aqua,Terra
ano,,
2001,NaN,10.56
2002,13.62,10.46
2003,13.60,10.44
2004,13.62,10.43
2005,13.59,10.43
2006,13.62,10.46
2007,13.65,10.43
2008,13.66,10.46
2009,13.65,10.48


**Resultado do teste:** a hora local média ficou estável até 2021 (Aqua ≈ 13,6 h; Terra ≈ 10,45 h) e depois **derivou**: em 2025, Aqua ≈ 14,9 h e Terra ≈ 9,3 h.
**Consequência:** a partir de 2022 o MODIS observa em outro horário do dia (o ciclo diário do fogo muda as contagens). **Calibrar a conversão só com 2012–2021.**

---
## Passo 5 — Qualidade e contexto
### 5a. Zero × ausência de observação
Um mês com 0 focos pode significar *sem fogo* **ou** *sem dado*. Testamos quantos dias do ano têm pelo menos um registro.

In [10]:
q(f'''SELECT ano, max(m) dias_modis, max(v) dias_viirs FROM (
        SELECT year(CAST(acq_date AS DATE)) ano, count(DISTINCT acq_date) m, 0 v FROM {M} GROUP BY 1
        UNION ALL SELECT year(CAST(acq_date AS DATE)), 0, count(DISTINCT acq_date) FROM {V} GROUP BY 1) GROUP BY 1 ORDER BY 1''')

,ano,dias_modis,dias_viirs
0,2001,347,0
1,2002,355,0
2,2003,365,0
3,2004,366,0
4,2005,365,0
5,2006,365,0
6,2007,365,0
7,2008,366,0
8,2009,365,0
9,2010,365,0


**Leitura:** dias faltando = **falta de observação ou de dado** (satélite fora do ar, arquivo incompleto), não zero fogo.
**Resultado:** MODIS 2001 (347 dias) e 2002 (355) e VIIRS 2012 (346, começa em 20/jan), 2022 (349) e 2024 (347) têm dias sem nenhum registro → **falta de observação/dado**, não zero fogo. Meses afetados devem ser marcados no calendário.
**Limitação:** o FIRMS só publica detecções; dias sem nenhuma detecção no Brasil inteiro são raros, então um dia ausente é forte sinal de falta de dado. Na grade mensal, **célula sem linha = 0 detecção ou não observado** — não dá para separar só com estes arquivos.

### 5b. Chave de registro e duplicatas

In [11]:
chave = '(latitude, longitude, acq_date, acq_time, satellite)'
for nome, src in [('MODIS', M), ('VIIRS', V)]:
    r = q(f'SELECT count(*) - count(DISTINCT {chave}) rep FROM {src}').rep[0]
    print(f'{nome}: registros com a mesma chave {chave}: {r}')
q(f'''SELECT * FROM {V} WHERE {chave} IN (
       SELECT {chave} FROM {V} GROUP BY ALL HAVING count(*) > 1) ORDER BY acq_date, latitude''')

MODIS: registros com a mesma chave (latitude, longitude, acq_date, acq_time, satellite): 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

VIIRS: registros com a mesma chave (latitude, longitude, acq_date, acq_time, satellite): 5


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,latitude,longitude,acq_date,acq_time,satellite,confidence,frp,daynight,type
0,-11.56517,-54.65517,2017-07-23,1720,N,n,10.49,D,0
1,-11.56517,-54.65517,2017-07-23,1720,N,n,12.12,D,0
2,-11.91423,-54.06617,2020-05-29,1701,N,h,89.50,D,0
3,-11.91423,-54.06617,2020-05-29,1701,N,l,114.40,D,0
4,-16.95864,-56.29676,2020-08-19,1803,N,n,9.22,D,0
5,-16.95864,-56.29676,2020-08-19,1803,N,n,10.88,D,0
6,-9.05636,-72.36766,2022-10-10,1845,N,n,11.06,D,0
7,-9.05636,-72.36766,2022-10-10,1845,N,n,11.14,D,0
8,-15.90648,-47.32532,2025-09-14,1741,N,n,7.78,D,0
9,-15.90648,-47.32532,2025-09-14,1741,N,n,19.92,D,0


**Critério:** chave = posição + data + hora + satélite (o que identifica um pixel numa passagem).
**Leitura:** se as linhas repetidas diferem em `frp`/`confidence`, **não são duplicatas exatas** — são registros distintos com a mesma chave (provável sobreposição de grânulos). Não remover sem decidir a regra; impacto: desprezível (poucas linhas em 19 mi).

### 5c. Fuso horário
`acq_date`/`acq_time` são **UTC** nos dois. Passagem noturna do Terra (~22:30 local ≈ 01:30 UTC) cai no dia seguinte em UTC.

In [12]:
q(f'''SELECT satellite, daynight, round(avg((acq_time::INT//100 < 6)::INT)*100, 1) pct_utc_madrugada
      FROM {M} GROUP BY ALL ORDER BY 1, 2''')

,satellite,daynight,pct_utc_madrugada
0,Aqua,D,0.0
1,Aqua,N,95.1
2,Terra,D,0.0
3,Terra,N,99.3


---
## Passo 6 — Transformações que vamos aplicar (registradas)
| Transformação | Regra | Justificativa |
|---|---|---|
| Filtro de tipo | `type = 0` | Remove fontes estáticas, vulcões e offshore (não são queimadas de vegetação) |
| Filtro de confiança | MODIS `confidence ≥ 30`; VIIRS `n` ou `h` | Exclui detecções de baixa confiança nos dois; equivalência entre os limiares é **hipótese** |
| Agregação espacial | célula 0,25° | Elimina a comparação ponto a ponto, que não é válida (Passo 1b) |
| Agregação temporal | mês, em **UTC** | Mesma regra nos dois sensores |
| Dados originais | `data/raw` preservado, sem alteração | Rastreabilidade |
| Duplicatas | **não removidas** | Chave repetida não é duplicata exata (5b) |

---
## Passo 7 — Tabela final de identificação (campo a campo)

In [13]:
final = pd.DataFrame([
 ('latitude/longitude', 'Equivalente sob critério', 'unidade e referência', 'graus, centro do pixel; 4 × 5 casas decimais; pixel 1 km × 375 m (1a, 3a)', 'mesma unidade, mas cada ponto cobre área diferente', 'comparar só agregado em células'),
 ('acq_date',           'Idêntico',                 'formato, fuso, regra',  'AAAA-MM-DD UTC nos dois (3a)', 'mesmo formato e convenção', 'UTC ≠ data local (5c)'),
 ('acq_time',           'Idêntico (formato/regra)', 'formato, fuso, resolução', 'HHMM UTC, 1 min, nos dois (3a)', 'mesmo formato; valores diferem por horário de passagem', 'não é a hora do fogo, é a da passagem'),
 ('daynight',           'Idêntico',                 'regra de atribuição',   'D só ~9–15h locais nos dois (notebook 00, 1.4b)', 'mesma regra (SZA 85°, dados standard)', 'NRT usa outra regra (órbita)'),
 ('instrument',         'Idêntico',                 'rótulo',                'MODIS / VIIRS', 'só identifica o sensor', '—'),
 ('satellite',          'Distinto',                 'domínio',               'Terra/Aqua × N (3a)', 'satélites diferentes', 'mapear para nomes'),
 ('type',               'Equivalente sob critério', 'códigos e uso',         'mesmos códigos; % type=0 ≈ 98,7% nos dois; offshore 133 × 29.251', 'mesma semântica, classificação por sensor', 'usar só como filtro'),
 ('confidence',         'Não comparável diretamente','escala',               '0–100 × l/n/h, regra muda dia/noite (3c)', 'algoritmos diferentes', 'equivalência de limiar é hipótese'),
 ('frp',                'Não comparável por ponto', 'unidade e área',        'MW nos dois; mediana 22,8 × 5,7 (notebook 00)', 'potência do pixel, pixel de tamanho diferente', 'comparável só somado por área/evento'),
 ('scan, track',        'Distinto',                 'domínio',               'faixas de tamanho de pixel diferentes (1a)', 'geometria do sensor', 'explicativo, não entra na série'),
 ('version',            'Distinto entre sensores; inconclusivo no MODIS', 'domínio', 'VIIRS sempre 2; MODIS 6.2 / 6.03 / 61.03 conforme o ano (3b)', 'produtos diferentes', 'verificar se o MODIS mistura coleções'),
 ('brightness / bright_ti4', 'Não comparável diretamente', 'banda espectral', 'MODIS 4 µm × VIIRS I4 3,74 µm', 'bandas e pixels diferentes', 'não usar'),
 ('bright_t31 / bright_ti5', 'Não comparável diretamente', 'banda espectral', 'MODIS 11 µm × VIIRS I5 11,45 µm', 'bandas e pixels diferentes', 'não usar'),
 ('registro (linha)',   'Distinto',                 'identidade',            '1 linha = 1 pixel numa passagem; sem chave comum (1b, Passo 2)', 'produtos independentes', '—'),
 ('evento (fogo)',      'Equivalente sob critério', 'proximidade espaço-tempo', 'pares Aqua×VIIRS a ≤1 km com poucos minutos de diferença (1b, 1c)', 'mesmo fogo visto pelos dois', 'tolerância testada 0,5–5 km'),
], columns=['item', 'classificação', 'critério', 'evidência', 'justificativa', 'limitações'])
final

,item,classificação,critério,evidência,justificativa,limitações
0,latitude/longitude,Equivalente sob critério,unidade e referência,"graus, centro do pixel; 4 × 5 casas decimais; ...","mesma unidade, mas cada ponto cobre área difer...",comparar só agregado em células
1,acq_date,Idêntico,"formato, fuso, regra",AAAA-MM-DD UTC nos dois (3a),mesmo formato e convenção,UTC ≠ data local (5c)
2,acq_time,Idêntico (formato/regra),"formato, fuso, resolução","HHMM UTC, 1 min, nos dois (3a)",mesmo formato; valores diferem por horário de ...,"não é a hora do fogo, é a da passagem"
3,daynight,Idêntico,regra de atribuição,"D só ~9–15h locais nos dois (notebook 00, 1.4b)","mesma regra (SZA 85°, dados standard)",NRT usa outra regra (órbita)
4,instrument,Idêntico,rótulo,MODIS / VIIRS,só identifica o sensor,—
5,satellite,Distinto,domínio,Terra/Aqua × N (3a),satélites diferentes,mapear para nomes
6,type,Equivalente sob critério,códigos e uso,"mesmos códigos; % type=0 ≈ 98,7% nos dois; off...","mesma semântica, classificação por sensor",usar só como filtro
7,confidence,Não comparável diretamente,escala,"0–100 × l/n/h, regra muda dia/noite (3c)",algoritmos diferentes,equivalência de limiar é hipótese
8,frp,Não comparável por ponto,unidade e área,"MW nos dois; mediana 22,8 × 5,7 (notebook 00)","potência do pixel, pixel de tamanho diferente",comparável só somado por área/evento
9,"scan, track",Distinto,domínio,faixas de tamanho de pixel diferentes (1a),geometria do sensor,"explicativo, não entra na série"


### Conclusão
- **Idênticos (reaproveita direto):** `acq_date`, `acq_time` (formato), `daynight`, `instrument`.
- **Equivalentes sob critério:** posição (agregando em célula), `type` (como filtro), o *evento de fogo* (por proximidade).
- **Não comparáveis diretamente:** `confidence`, `frp` por ponto, temperaturas de brilho.
- **Até onde vale:** Brasil, arquivos *standard* anuais do FIRMS (não NRT), MODIS C6.1 e VIIRS S-NPP v2. Contagem de pontos **não é** número de incêndios nem área queimada.